# KV260 — Năng lượng & bộ nhớ của bước hiệu chỉnh trên ARM (việc 2, bài journal)
Chép thêm vào thư mục `F2A_BOARD_RUN` trên board (cạnh các file đã có):
- `lib_ecg_f2.py` (lấy từ `05_board/ecg_f2_seed3/`) — dùng lớp `PowerMonitor` (INA260, đếm 1 lần) và `NumpyF2`
- `f2_params.npz` (= `02_quant/f2_params_seed3.npz`, đổi tên) — để ước lượng chi phí fine-tune toàn mạng trên ARM

**Trước khi đo:** tắt cập nhật tự động (`sudo systemctl stop unattended-upgrades apt-daily.timer apt-daily-upgrade.timer`), đóng các notebook khác.
Kết quả lưu vào `f2a_adapt_energy.json`.

In [ ]:
import time, json, gc, tracemalloc, numpy as np
from pynq import Overlay, Clocks
import lib_ecg_f2a as L, pa_adapt as pa, lib_ecg_f2 as LF
ol = Overlay('ecg_f2a.bit'); Clocks.fclk0_mhz = 100; print('fclk0:', Clocks.fclk0_mhz, 'MHz')
regs = L.parse_hw_header('xecg_f2a_top_hw.h'); ip = L.ECGF2A(ol.ecg_f2a_top_0.mmio, regs)
W_G, B_G = L.parse_global_head('weights_f2a.h'); ip.write_head(W_G, B_G)
D = np.load('f2a_board_data.npz'); hp = json.loads(str(D['best_hp']))
hp['head_scratch'] = dict(lr=0.5, epochs=150)          # chọn trên DS1 ở exp4
beats, rrs, y, rec, t = D['beats_q'], D['rr_q'], D['y'], D['rec'], D['t']
W0 = W_G.reshape(4, 32).T / 128.0; b0 = B_G / 16384.0
_, E_all, _ = ip.infer_batch(beats, rrs)               # emb của mọi nhịp, lấy từ IP
_, EA, _ = ip.infer_batch(D['anc_beats_q'], D['anc_rr_q'])
pa.CURRENT_ANCHORS = (EA / 16.0, D['anc_y'])
cal = pa.split_first(t, rec, 5); RECS = np.unique(rec)
print('Nhịp hiệu chỉnh:', cal.sum(), '| số bản ghi:', len(RECS))

In [ ]:
def adapt_head_scratch(E, y, W0, b0, lr=0.5, epochs=300, cap=10.0, **kw):
    cnt = np.bincount(y, minlength=pa.NC).astype(float)
    cw = np.where(cnt > 0, np.minimum(cnt.sum() / (pa.NC * np.maximum(cnt, 1)), cap), 0.0)
    sw = cw[y] / cw[y].sum(); s = 1.0 / (E.std() + 1e-9); Es = E * s
    W = np.zeros_like(W0); b = np.zeros_like(b0); b[cnt == 0] = -1e3; Y = np.eye(pa.NC)[y]
    for _ in range(epochs):
        Z = Es @ W + b; Z -= Z.max(1, keepdims=True); P = np.exp(Z); P /= P.sum(1, keepdims=True)
        G = (P - Y) * sw[:, None]; W -= lr * Es.T @ G; b -= lr * G.sum(0) * (cnt > 0)
    return W * s, b
pa.METHODS['head_scratch'] = adapt_head_scratch
CAL = {r: (E_all[(rec == r) & cal] / 16.0, y[(rec == r) & cal]) for r in RECS}   # dữ liệu 5 phút đầu, chuẩn bị sẵn
def adapt_all(mth):
    for r in RECS:
        Ec, yc = CAL[r]
        W, b = pa.METHODS[mth](Ec, yc, W0, b0, **hp[mth])
        pa.head_to_registers(W, b)

## 1. Công suất nền (board rảnh) — trước

In [ ]:
pm = LF.PowerMonitor(); idle0 = pm.baseline(seconds=10)

## 2. Năng lượng hiệu chỉnh: lặp lại cả 22 bản ghi cho tới ≥ 15 s, lấy mẫu công suất mỗi 50 ms

In [ ]:
RES = {}
for mth in ['offset_int', 'head_scratch', 'gradient_anchor']:
    gc.collect(); time.sleep(2)
    _, st = pm.measure(lambda: adapt_all(mth), period=0.05, min_seconds=15)
    n = st['reps'] * len(RECS)
    RES[mth] = dict(seconds=st['seconds'], reps=st['reps'], n_records=n, p_mean_w=st['p_mean_w'],
                    t_per_record_ms=st['seconds'] / n * 1e3)
    print(f"{mth:16s} {RES[mth]['t_per_record_ms']:8.2f} ms/bản ghi | P_mean {st['p_mean_w']:.3f} W | {st['reps']} vòng")

## 3. Công suất nền — sau; tính năng lượng tăng thêm trên mỗi bản ghi

In [ ]:
idle1 = pm.baseline(seconds=10); idle = (idle0 + idle1) / 2
for mth, r in RES.items():
    r['p_active_w'] = r['p_mean_w'] - idle
    r['energy_mJ_per_record'] = r['p_active_w'] * r['seconds'] / r['n_records'] * 1e3
    print(f"{mth:16s} {r['t_per_record_ms']:8.2f} ms | +{r['p_active_w']:.3f} W | {r['energy_mJ_per_record']:.3f} mJ/bản ghi")
print('idle trước/sau:', round(idle0, 3), round(idle1, 3), 'W')

## 4. Bộ nhớ: đỉnh cấp phát Python khi hiệu chỉnh 1 bản ghi (lớn nhất trong 22) + bộ đệm cố định

In [ ]:
MEM = {}
for mth in ['offset_int', 'head_scratch', 'gradient_anchor']:
    peaks = []
    for r in RECS:
        Ec, yc = CAL[r]; tracemalloc.start()
        W, b = pa.METHODS[mth](Ec, yc, W0, b0, **hp[mth])
        peaks.append(tracemalloc.get_traced_memory()[1]); tracemalloc.stop()
    MEM[mth] = dict(peak_kB_max=max(peaks) / 1024, peak_kB_mean=float(np.mean(peaks)) / 1024)
n_cal_max = max(len(CAL[r][1]) for r in RECS)
BUF = dict(emb_cal_int8_kB=n_cal_max * 32 / 1024, labels_cal_kB=n_cal_max / 1024,
           anchors_kB=(100 * 32 + 100) / 1024, head_regs_B=128 + 4 * 4)
print(MEM); print(BUF)

## 5. Ước lượng chi phí fine-tune TOÀN MẠNG trên ARM (để so sánh)
Đo thời gian + công suất một lượt suy luận float32 NumPy trên các nhịp 5 phút đầu; huấn luyện ≈ 3× suy luận (thuận + ngược) × số epoch (30, chọn trên DS1).

In [ ]:
nf = LF.NumpyF2('f2_params.npz')
r0 = RECS[np.argmax([len(CAL[r][1]) for r in RECS])]; m0 = (rec == r0) & cal
Xf = beats[m0].astype(np.float32) / 64.0; Rf = rrs[m0].astype(np.float32) / 32.0
_, st = pm.measure(lambda: nf.float32(Xf, Rf), period=0.05, min_seconds=15)
t_fwd = st['seconds'] / (st['reps'] * m0.sum()); p_fwd = st['p_mean_w'] - idle
n_mean = np.mean([len(CAL[r][1]) for r in RECS]); EP = 30
FT = {}
for name, extra in [('ft_full', 0), ('ft_full_anchor', 100)]:
    tt = 3 * t_fwd * (n_mean + extra) * EP
    FT[name] = dict(t_est_s_per_record=tt, energy_est_J_per_record=tt * p_fwd)
print(f'float32 thuận: {t_fwd * 1e3:.3f} ms/nhịp, +{p_fwd:.3f} W'); print(FT)

In [ ]:
json.dump(dict(idle_w=[idle0, idle1], adapt=RES, memory=MEM, buffers=BUF, ft_estimate=FT,
               t_fwd_ms=t_fwd * 1e3, p_fwd_w=p_fwd, fclk_mhz=Clocks.fclk0_mhz), open('f2a_adapt_energy.json', 'w'), indent=1, default=float)
print('Đã lưu f2a_adapt_energy.json')